In [22]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Dropout, MultiHeadAttention, Add
from tensorflow.keras.callbacks import EarlyStopping

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
DB_NAME = "../../nba_data.db"
DB_URI = f"sqlite:///{DB_NAME}"
engine = create_engine(DB_URI, echo=False)

# Output CSV file for results
OUTPUT_CSV = "nba_points_predictions_transformer.csv"

# ------------------------------------------------------------
# Load Data & Sort
# ------------------------------------------------------------
try:
    df = pd.read_sql("SELECT * FROM player_game_features", engine)
    print(f"Loaded data with shape: {df.shape}")
except Exception as e:
    print(f"Error loading data from database: {e}")
    exit(1)

# Ensure data is sorted by player and date
df = df.sort_values(by=["player_id", "game_date"])

# Extract the season or year from 'game_date'
df['game_year'] = pd.to_datetime(df['game_date']).dt.year

# Features and target
features = ["player_id", "pts", "min", "fgm", "fga", "pts_per_min", "fg_pct"]
target = "pts"

# Drop rows with missing values in features or target
df = df.dropna(subset=features + [target])

if df.empty:
    print("No data available after dropping missing values.")
    exit(1)

print(f"Data after cleaning: {df.shape}")

# ------------------------------------------------------------
# Helper Function: Create Sequences (Fixed Max Length)
# ------------------------------------------------------------
def create_player_sequences_fixed_length(data, target, player_column, max_length):
    """
    Create sequences of all past games for each player, then pad them to 'max_length'.
    Returns padded sequences, targets, and original indices for mapping.
    """
    X_list, y_list, indices_list, player_ids_list = [], [], [], []
    
    # Group by player_id
    for p_id, group in data.groupby(player_column):
        if len(group) < 2:  # Skip players with less than 2 games (no sequence possible)
            continue
        # Convert features to np.array
        player_features = group.drop(columns=[player_column]).values
        # Pull the corresponding target values
        player_target = target[group.index].values
        # Store original indices to map back later
        player_indices = group.index.values

        # Build sequences from length=1 up to the current index
        for i in range(1, len(player_features)):
            seq = player_features[:i]  # All past games up to (not including) i
            X_list.append(seq)
            y_list.append(player_target[i])  # The target at index i
            indices_list.append(player_indices[i])  # Store the index for mapping
            player_ids_list.append(p_id)  # Store player_id for mapping

    if not X_list:
        print("No sequences generated.")
        return np.array([]), np.array([]), [], []

    # Now, pad/truncate each sequence to 'max_length'
    num_features = X_list[0].shape[1] if X_list else 0
    X_padded = np.zeros((len(X_list), max_length, num_features), dtype=np.float32)

    for i, seq in enumerate(X_list):
        seq_len = len(seq)
        if seq_len <= max_length:
            # Put seq at the end, zeros at the front
            X_padded[i, max_length - seq_len:, :] = seq
        else:
            # If sequence is longer than max_length, truncate from the front
            X_padded[i, :, :] = seq[-max_length:]

    return X_padded, np.array(y_list), indices_list, player_ids_list

# ------------------------------------------------------------
# Helper Function: Build Transformer Model
# ------------------------------------------------------------
def build_transformer_model(input_shape, num_heads=2, dff=64, dropout_rate=0.1):
    """
    Build a simplified Transformer model for regression with a custom encoder.
    """
    inputs = Input(shape=input_shape)
    seq_len = input_shape[0]
    d_model = input_shape[1]

    # Simplified positional encoding (fixed shape)
    position = tf.range(start=0, limit=seq_len, delta=1, dtype=tf.float32)
    position = tf.expand_dims(position, -1)
    position_embedding = Dense(d_model, activation=None)(position)
    position_embedding = tf.expand_dims(position_embedding, 0)
    x = inputs + position_embedding  # Broadcasting to match batch dimension

    # Transformer Encoder Block (simplified with fewer heads and smaller dimensions)
    attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)(x, x)
    attention_output = Dropout(dropout_rate)(attention_output)
    out1 = Add()([x, attention_output])
    out1 = LayerNormalization(epsilon=1e-6)(out1)
    
    # Feed Forward Network
    ffn_output = Dense(dff, activation='relu')(out1)
    ffn_output = Dense(d_model)(ffn_output)
    ffn_output = Dropout(dropout_rate)(ffn_output)
    out2 = Add()([out1, ffn_output])
    out2 = LayerNormalization(epsilon=1e-6)(out2)
    
    # Use the last timestep as the summary of the sequence
    out2 = out2[:, -1, :]
    
    # Output layer for regression
    outputs = Dense(1)(out2)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# ------------------------------------------------------------
# Rolling/Expanding Window Validation
# ------------------------------------------------------------
available_years = sorted(df['game_year'].unique())
print("Available Years in Data:", available_years)

training_window = 4
mae_scores = []
rmse_scores = []
years_tested = []
all_results = []  # List to store results for CSV

for validate_year in available_years:
    start_train_year = validate_year - training_window
    if start_train_year < available_years[0]:
        print(f"Skipping {validate_year}: Not enough prior data for training window.")
        continue
    if not all(y_ in available_years for y_ in range(start_train_year, validate_year)):
        print(f"Skipping {validate_year}: Missing intermediate years in data.")
        continue

    # Build train/val splits
    train_mask = (df['game_year'] >= start_train_year) & (df['game_year'] < validate_year)
    val_mask = (df['game_year'] == validate_year)

    train_data = df[train_mask]
    val_data = df[val_mask]

    if len(train_data) == 0 or len(val_data) == 0:
        print(f"Skipping {validate_year}: Empty training or validation data.")
        continue

    print(f"Processing {validate_year}: Train data shape {train_data.shape}, Val data shape {val_data.shape}")

    # Scale only the feature columns except "player_id"
    scaler = MinMaxScaler()
    train_features = train_data[features].drop(columns=["player_id"])
    val_features = val_data[features].drop(columns=["player_id"])
    if train_features.empty or val_features.empty:
        print(f"Skipping {validate_year}: Empty feature data after dropping player_id.")
        continue

    scaled_features_train = scaler.fit_transform(train_features)
    scaled_features_val = scaler.transform(val_features)

    # Create scaled dataframes with 'player_id' re-attached
    train_scaled = pd.DataFrame(scaled_features_train, 
                                index=train_data.index, 
                                columns=features[1:])
    train_scaled["player_id"] = train_data["player_id"].values

    val_scaled = pd.DataFrame(scaled_features_val, 
                              index=val_data.index, 
                              columns=features[1:])
    val_scaled["player_id"] = val_data["player_id"].values

    # Find max sequence lengths so we can pad to a single max_length
    def find_player_longest_sequence(data_df, id_col="player_id"):
        max_len = 0
        for _, group in data_df.groupby(id_col):
            length = len(group)
            max_len = max(max_len, length - 1)
        return max_len if max_len > 0 else 1

    max_len_train = find_player_longest_sequence(train_scaled, "player_id")
    max_len_val = find_player_longest_sequence(val_scaled, "player_id")
    max_len_both = max(max_len_train, max_len_val)
    if max_len_both < 1:
        print(f"Skipping {validate_year}: Sequence length is too short to form sequences.")
        continue

    # Cap max_len_both to avoid excessive memory usage
    max_len_both = min(max_len_both, 50)
    print(f"Using max sequence length: {max_len_both}")

    # Create sequences
    X_train, y_train, _, _ = create_player_sequences_fixed_length(
        train_scaled, train_data[target], "player_id", max_len_both
    )
    X_val, y_val, val_indices, val_player_ids = create_player_sequences_fixed_length(
        val_scaled, val_data[target], "player_id", max_len_both
    )

    if len(X_train) == 0 or len(X_val) == 0:
        print(f"Skipping {validate_year}: No sequences generated for training or validation.")
        continue

    print(f"Train sequences: {X_train.shape}, Val sequences: {X_val.shape}")

    # Build the Transformer model
    num_features = X_train.shape[2] if len(X_train.shape) == 3 else 0
    if num_features == 0:
        print(f"Skipping {validate_year}: Invalid feature dimension in training data.")
        continue

    input_shape = (max_len_both, num_features)
    try:
        model = build_transformer_model(input_shape=input_shape)
        model.summary()
    except Exception as e:
        print(f"Error building model for year {validate_year}: {e}")
        continue

    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Train the model
    try:
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=20, batch_size=16,  # Reduced for stability
            verbose=1, callbacks=[early_stop]
        )
    except Exception as e:
        print(f"Error training model for year {validate_year}: {e}")
        continue

    # Predict on the validation set
    try:
        y_pred = model.predict(X_val, batch_size=16).flatten()
        if len(y_pred) != len(y_val):
            print(f"Warning: Prediction length mismatch for year {validate_year}. Adjusting.")
            y_pred = y_pred[:len(y_val)]
    except Exception as e:
        print(f"Error predicting for year {validate_year}: {e}")
        continue

    # Calculate metrics
    mae = mean_absolute_error(y_val, y_pred)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    years_tested.append(validate_year)

    print(f"Validation Year: {validate_year}")
    print(f"Train Years: {start_train_year} to {validate_year-1}")
    print(f"MAE:  {mae:.2f}")
    print(f"RMSE: {rmse:.2f}\n")

    # Create a DataFrame for the validation results
    try:
        # Ensure lengths match for DataFrame creation
        if len(val_indices) != len(y_val):
            print(f"Warning: Index length mismatch for year {validate_year}. Using fallback.")
            val_indices = val_indices[:len(y_val)] if len(val_indices) > len(y_val) else val_indices + [None] * (len(y_val) - len(val_indices))
        if len(val_player_ids) != len(y_val):
            val_player_ids = val_player_ids[:len(y_val)] if len(val_player_ids) > len(y_val) else val_player_ids + ['N/A'] * (len(y_val) - len(val_player_ids))

        # Safely extract game_date with fallback
        game_dates = []
        for idx in val_indices:
            if idx in val_data.index:
                game_dates.append(val_data.loc[idx, 'game_date'])
            else:
                game_dates.append('N/A')

        val_results = pd.DataFrame({
            'validation_year': validate_year,
            'index': val_indices,
            'player_id': val_player_ids,
            'game_date': game_dates,
            'actual_pts': y_val,
            'predicted_pts': y_pred,
            'mae': mae,
            'rmse': rmse
        })
        # Classify as over/under based on actual points (for reference in results)
        val_results['over_under'] = np.where(
            val_results['predicted_pts'] > val_results['actual_pts'], 'Over', 'Under'
        )
        all_results.append(val_results)
    except Exception as e:
        print(f"Error creating results DataFrame for year {validate_year}: {e}")
        continue

# Concatenate all results into a single DataFrame
if all_results:
    try:
        final_results = pd.concat(all_results, ignore_index=True)
        # Save to CSV
        final_results.to_csv(OUTPUT_CSV, index=False)
        print(f"Results saved to {OUTPUT_CSV}")
    except Exception as e:
        print(f"Error saving results to CSV: {e}")
else:
    print("No results to save.")

Loaded data with shape: (201805, 23)
Data after cleaning: (201805, 24)
Available Years in Data: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Skipping 2015: Not enough prior data for training window.
Skipping 2016: Not enough prior data for training window.
Skipping 2017: Not enough prior data for training window.
Skipping 2018: Not enough prior data for training window.
Processing 2019: Train data shape (90010, 24), Val data shape (25100, 24)
Using max sequence length: 50
Train sequences: (89217, 50, 6), Val sequences: (24490, 50, 6)


Model: "functional_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_26 (InputLayer)   │ (None, 50, 6)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_42 (Add)                  │ (None, 50, 6)             │               0 │ input_layer_26[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multi_head_attention_14       │ (None, 50, 6)             │             168 │ add_42[0][0], add_42[0][0] │
│ (MultiHeadAttention)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_43 (Dropout)          │ (None, 50, 6)             │               0 │ multi_head_attention_14[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_43 (Add)                  │ (None, 50, 6)             │               0 │ add_42[0][0],              │
│                               │                           │                 │ dropout_43[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_28        │ (None, 50, 6)             │              12 │ add_43[0][0]               │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_53 (Dense)              │ (None, 50, 64)            │             448 │ layer_normalization_28[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_54 (Dense)              │ (None, 50, 6)             │             390 │ dense_53[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_44 (Dropout)          │ (None, 50, 6)             │               0 │ dense_54[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_44 (Add)                  │ (None, 50, 6)             │               0 │ layer_normalization_28[0]… │
│                               │                           │                 │ dropout_44[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_29        │ (None, 50, 6)             │              12 │ add_44[0][0]               │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ get_item_14 (GetItem)         │ (None, 6)                 │               0 │ layer_normalization_29[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_55 (Dense)              │ (None, 1)                 │               7 │ get_item_14[0][0]          │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,037 (4.05 KB)

 Trainable params: 1,037 (4.05 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_30 (InputLayer)   │ (None, 50, 6)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_54 (Add)                  │ (None, 50, 6)             │               0 │ input_layer_30[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multi_head_attention_18       │ (None, 50, 6)             │             168 │ add_54[0][0], add_54[0][0] │
│ (MultiHeadAttention)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_55 (Dropout)          │ (None, 50, 6)             │               0 │ multi_head_attention_18[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_55 (Add)                  │ (None, 50, 6)             │               0 │ add_54[0][0],              │
│                               │                           │                 │ dropout_55[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_36        │ (None, 50, 6)             │              12 │ add_55[0][0]               │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_69 (Dense)              │ (None, 50, 64)            │             448 │ layer_normalization_36[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_70 (Dense)              │ (None, 50, 6)             │             390 │ dense_69[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_56 (Dropout)          │ (None, 50, 6)             │               0 │ dense_70[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_56 (Add)                  │ (None, 50, 6)             │               0 │ layer_normalization_36[0]… │
│                               │                           │                 │ dropout_56[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_37        │ (None, 50, 6)             │              12 │ add_56[0][0]               │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ get_item_18 (GetItem)         │ (None, 6)                 │               0 │ layer_normalization_37[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_71 (Dense)              │ (None, 1)                 │               7 │ get_item_18[0][0]          │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,037 (4.05 KB)

 Trainable params: 1,037 (4.05 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
6037/6037 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 84.4445 - mae: 7.0804 - val_loss: 82.3475 - val_mae: 7.1038
Epoch 2/20
6037/6037 ━━━━━━━━━━━━━━━━━━━━ 33s 5ms/step - loss: 70.0145 - mae: 6.5602 - val_loss: 63.9589 - val_mae: 5.8057
Epoch 3/20
6037/6037 ━━━━━━━━━━━━━━━━━━━━ 6214s 1s/step - loss: 45.4026 - mae: 5.1995 - val_loss: 66.5732 - val_mae: 5.8958
Epoch 4/20
6037/6037 ━━━━━━━━━━━━━━━━━━━━ 34s 6ms/step - loss: 44.2678 - mae: 5.1373 - val_loss: 48.2480 - val_mae: 5.2818
Epoch 5/20
6037/6037 ━━━━━━━━━━━━━━━━━━━━ 33s 5ms/step - loss: 42.8615 - mae: 5.0363 - val_loss: 52.5923 - val_mae: 5.3875
Epoch 6/20
6037/6037 ━━━━━━━━━━━━━━━━━━━━ 33s 5ms/step - loss: 42.1897 - mae: 5.0187 - val_loss: 58.2398 - val_mae: 5.5737
Epoch 7/20
6037/6037 ━━━━━━━━━━━━━━━━━━━━ 33s 5ms/step - loss: 42.5013 - mae: 5.0306 - val_loss: 48.1644 - val_mae: 5.2162
Epoch 8/20
6037/6037 ━━━━━━━━━━━━━━━━━━━━ 33s 5ms/step - loss: 42.0052 - mae: 4.9934 - val_loss: 55.6430 - val_mae: 5.4490
Epoch 9/20
6037